# 3장 실습 — 최단경로 구현과 채점

이번 학기에 직접 짜는 셋 중 첫 번째입니다.

채울 파일은 `labs/ch03_dijkstra.py` 입니다. 이 노트북이 아니라 그 파일을 고칩니다.
노트북은 채운 것을 바로 확인하는 용도입니다. 편집기에서 파일을 열어 두고,
함수 하나를 채울 때마다 이 노트북의 해당 셀을 다시 실행하세요.

순서는 이렇습니다.

1. 손으로 답을 아는 작은 그래프로 확인합니다
2. 하남시 도로망으로 옮깁니다
3. `NetworkX` 와 대조합니다
4. `check("ch03")` 으로 채점합니다

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect
from smartmob.viz import use_korean_font

use_korean_font()

import ch03_dijkstra as sol      # 여러분이 채우는 파일

`autoreload` 를 켜 두었으므로 `labs/ch03_dijkstra.py` 를 저장하면
이 노트북을 다시 시작하지 않아도 반영됩니다.

## 1. 손으로 답을 아는 그래프

노드 여섯 개짜리 그래프를 만듭니다. 종이에 그려 놓고 따라갈 수 있는 크기입니다.

```
       (600m)        (900m)
  A ────────────► B ────────────► C
  │               │               ▲
  │(2000m)        │(300m)         │(300m)
  │               ▼               │
  └──────────────►D───────────────┘

  E ────────────► F        (A 쪽과 이어져 있지 않습니다)
         (300m)
```

모든 도로가 시속 36km, 즉 초속 10m입니다. 그래서 거리를 10으로 나누면 초가 됩니다.

A에서 C로 가는 길은 셋입니다.

| 경로 | 거리 | 소요시간 |
|---|---|---|
| A → C | 2,000m | 200초 |
| A → B → C | 1,500m | 150초 |
| A → B → D → C | 1,200m | 120초 |

정답은 120초, 경로는 `[n1, n2, n4, n3]` 입니다.

In [ ]:
import pandas as pd

from smartmob.teaching.graph import RoadGraph

toy_nodes = pd.DataFrame([
    {"node_id": "n1", "lat": 37.5000, "lon": 127.2000},   # A
    {"node_id": "n2", "lat": 37.5030, "lon": 127.2000},   # B
    {"node_id": "n3", "lat": 37.5080, "lon": 127.2000},   # C
    {"node_id": "n4", "lat": 37.5050, "lon": 127.2010},   # D
    {"node_id": "n5", "lat": 37.5500, "lon": 127.2500},   # E
    {"node_id": "n6", "lat": 37.5520, "lon": 127.2500},   # F
])

toy_edges = pd.DataFrame([
    {"edge_id": "e1_f_1_2", "length": 600.0},
    {"edge_id": "e2_f_1_3", "length": 2000.0},
    {"edge_id": "e3_f_2_3", "length": 900.0},
    {"edge_id": "e4_f_2_4", "length": 300.0},
    {"edge_id": "e5_f_4_3", "length": 300.0},
    {"edge_id": "e6_f_5_6", "length": 300.0},
]).assign(highway="residential", free_flow_speed_kmh=36.0)

toy = RoadGraph.from_frames(toy_nodes, toy_edges, modes=("drive",))
toy

`labs/ch03_dijkstra.py` 의 `dijkstra` 와 `trace` 를 채우고 아래 셀을 실행합니다.
아직 안 채웠으면 "아직 구현하지 않았습니다"가 나옵니다. 정상입니다.

In [ ]:
banner("작은 그래프 확인")
try:
    seconds, path, settled = sol.dijkstra(toy, "n1", "n3")
    expect("A→C 소요시간(초)", seconds, 120.0, tol=0.01)
    expect("A→C 경로", path, ["n1", "n2", "n4", "n3"])
    print(f"    확정한 노드 {settled}개")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)

길이 없는 경우도 확인합니다. A에서 E로는 갈 수 없으므로 예외가 나야 합니다.
조용히 무한대를 돌려주면 시뮬레이터가 그 승객을 영원히 기다리게 만듭니다.

In [ ]:
try:
    sol.dijkstra(toy, "n1", "n5")
    print("[x] 예외를 던지지 않았습니다. 길이 없을 때는 예외를 내야 합니다")
except NotImplementedError:
    print("[ ] 아직 구현하지 않았습니다")
except Exception as exc:
    print(f"[v] 길이 없을 때 {type(exc).__name__} 를 냅니다 — {exc}")

## 2. 하남시 도로망으로 (교재 3.3)

작은 그래프에서 맞으면 실제 도로망으로 옮깁니다.
노드가 12,566개로 늘어날 뿐 알고리즘은 그대로입니다.

In [ ]:
from smartmob.data import load_road_graph

G = load_road_graph("hanam", modes=("drive",))
start = G.nearest_node(37.5393, 127.2148)    # 하남시청
goal = G.nearest_node(37.5606, 127.1930)     # 미사역

print("출발", start, G.coord[start])
print("도착", goal, G.coord[goal])

In [ ]:
try:
    seconds, path, settled = sol.dijkstra(G, start, goal)
    print(f"소요시간 {seconds / 60:.2f}분")
    print(f"거친 노드 {len(path):,}개")
    print(f"확정한 노드 {settled:,}개  (전체 {G.n_nodes:,}개의 {settled / G.n_nodes:.0%})")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

교재와 같은 값이 나와야 합니다. 5분 30초 안팎, 확정 노드 4,500개 남짓입니다.

눈여겨볼 것은 마지막 줄입니다. 83개 노드짜리 경로를 얻으려고 도로망의 3분의 1을 확정했습니다.
출발점에서 사방으로 고르게 퍼지기 때문입니다. 이것이 4절에서 A\*를 도입하는 이유입니다.

## 3. 반환 경로 검증 (교재 3.3)

소요시간이 맞아도 `prev` 를 잘못 채우면 경로가 끊어져 있을 수 있습니다.
엣지를 하나씩 되짚어 더한 값이 반환값과 같은지 봅니다.

In [ ]:
try:
    seconds, path, settled = sol.dijkstra(G, start, goal)
    total = 0.0
    for u, v in zip(path, path[1:]):
        edge = next((w for nb, w, _ in G.neighbors(u) if nb == v), None)
        assert edge is not None, f"{u} → {v} 엣지가 없습니다"
        total += edge
    expect("엣지를 다시 더한 값", round(total, 3), round(seconds, 3), tol=0.01)
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

## 4. `NetworkX` 와 대조 (교재 3.4)

코드가 실행되는 것과 결과가 맞는 것은 별개입니다.
남이 만든 구현과 30쌍을 맞춰 봅니다. 이것이 채점 기준이기도 합니다.

In [ ]:
import random

import networkx as nx

nxg = nx.DiGraph()
for u in G.nodes:
    for v, seconds_, _ in G.neighbors(u):
        if not nxg.has_edge(u, v) or nxg[u][v]["weight"] > seconds_:
            nxg.add_edge(u, v, weight=seconds_)

print(f"NetworkX 그래프 노드 {nxg.number_of_nodes():,}개, 엣지 {nxg.number_of_edges():,}개")

In [ ]:
rng = random.Random(42)
node_list = [n for n in G.nodes if G.adj[n]]

banner("NetworkX 대조 (10쌍)")
try:
    worst, compared = 0.0, 0
    for _ in range(10):
        u, v = rng.choice(node_list), rng.choice(node_list)
        try:
            want = nx.shortest_path_length(nxg, u, v, weight="weight")
        except nx.NetworkXNoPath:
            continue
        got, _, _ = sol.dijkstra(G, u, v)
        worst = max(worst, abs(got - want))
        compared += 1
    print(f"{compared}쌍 비교, 최대 오차 {worst:.6f}초")
    print("[v] 통과" if worst < 1e-6 else "[x] 값이 다릅니다. 힙에서 꺼낼 때 종료했는지 확인하세요")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

## 5. A\* (교재 3.5, 3.6)

A\*는 남은 소요시간의 하한을 우선순위에 더합니다.
이 실습에서는 직선거리를 도로망의 최고 속도로 나눈 값을 사용합니다.
실제보다 작아야(허용 가능해야) 최적해가 유지됩니다.

In [ ]:
print(f"이 도로망의 최고 속도 {G.max_speed_kmh():.0f} km/h")

banner("다익스트라와 A* 비교")
try:
    d_sec, d_path, d_settled = sol.dijkstra(G, start, goal)
    a_sec, a_path, a_settled = sol.astar(G, start, goal)

    expect("두 방법의 소요시간이 같다", round(a_sec, 6), round(d_sec, 6), tol=1e-6)
    print(f"    확정 노드  다익스트라 {d_settled:,}  →  A* {a_settled:,}"
          f"  ({1 - a_settled / d_settled:.0%} 감소)")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

하남시청과 미사역 사이에서는 확정 노드가 4,565개에서 3,571개로 줄어듭니다.
교재 3.6절의 무작위 30쌍에서는 중앙값이 7,320개에서 3,978.5개로 줄었습니다.
한 쌍의 결과만으로 두 알고리즘의 탐색량을 비교할 수는 없습니다.

확정 노드가 줄어도 실행시간은 늘어날 수 있습니다.
이 구현에서는 각 후보 노드의 하버사인 거리를 파이썬으로 계산합니다.
직접 재 봅니다.

In [ ]:
import time

try:
    for name, fn in [("다익스트라", sol.dijkstra), ("A*", sol.astar)]:
        t0 = time.perf_counter()
        for _ in range(5):
            fn(G, start, goal)
        print(f"{name:10s} {(time.perf_counter() - t0) / 5 * 1000:7.1f} ms")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

## 6. 채점

과제 채점에 사용하는 검사를 실행합니다. 모든 항목이 `PASS`인지 확인합니다.

In [ ]:
from check import check

report = check("ch03")

## 제출할 것

1. 채운 `labs/ch03_dijkstra.py`
2. 위 채점 셀의 출력(모든 항목 `PASS`)
3. 막혔던 지점과 어떻게 풀었는지 3~5줄

## 정리

- 미확정 노드 중 거리가 가장 작은 노드는 그 값으로 확정됩니다
- 도착 노드를 힙에 넣을 때가 아니라 꺼낼 때 끝냅니다
- 경로가 없으면 예외를 냅니다. 무한대를 반환하면 배차 단계에서 원인을 찾기 어렵습니다
- A\*는 확정 노드를 줄이지만 파이썬에서는 더 느릴 수 있습니다
- 4장 실습에서는 같은 그래프에 시간대별 속도를 넣어 경로가 어떻게 바뀌는지 봅니다